<h1>Imports and Packages<h1>

In [1]:
# Install required packages (run this cell first if you see import errors)
import sys
import subprocess
import importlib
import numpy as np
import pandas as pd
from typing import Union

packages = ['requests', 'beautifulsoup4']
for package in packages:
    try:
        importlib.import_module(package if package != 'beautifulsoup4' else 'bs4')
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet", "--user"])
        print(f"✓ {package} installed successfully")


In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://www.google.com"

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

<!DOCTYPE html>
<html itemscope="" itemtype="http://schema.org/WebPage" lang="en">
 <head>
  <meta content="Search the world's information, including webpages, images, videos and more. Google has many special features to help you find exactly what you're looking for." name="description"/>
  <meta content="noodp, " name="robots"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="/images/branding/googleg/1x/googleg_standard_color_128dp.png" itemprop="image"/>
  <title>
   Google
  </title>
  <script nonce="CKW2JmuEEVe6R4e5RJP0SA">
   (function(){var _g={kEI:'HjN5adWJIo3k2roP_Z__mA8',kEXPI:'0,203005,1101198,2935842,48791,30022,16105,344796,219811,27509,5285613,3,36811432,25228681,123988,28395,8939,3003,53220,39775,12269,48981,14068,23259,3286,34514,28333,48296,17000,14019,7707,7,33385,4719,21511,2863,1,726,2156,3088,7010,9747,28574,17766,21838,4430,6292,6436,9743,2646,13254,7475,2,1,3514,14505,1383,1,2,1515,3355,197,570,5243,2,4205,1,16100,2,873,2,22

<h1>Locations in Florida<h1>

In [3]:
from functools import lru_cache

# Florida Cities
FL_CITIES = {
    "Miami": (25.7617, -80.1918),
    "Orlando": (28.5383, -81.3792),
    "Tampa": (27.9506, -82.4572),
    "Jacksonville": (30.3322, -81.6557),
    "Tallahassee": (30.4383, -84.2807),
    "Fort Lauderdale": (26.1224, -80.1373),
    "Sarasota": (27.3364, -82.5307),
    "Naples": (26.1420, -81.7948),
    "Pensacola": (30.4213, -87.2169),
    "Key West": (24.5551, -81.7800),
}

# NWS asks for a User-Agent with contact info (email or website)
NWS_HEADERS = {
    "User-Agent": "FloridaWeatherAdvisor/1.0 (contact: yashkode@gmail.com)",
    "Accept": "application/geo+json",
}

def _get_json(url: str) -> dict:
    r = requests.get(url, headers=NWS_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=256)
def points_lookup(lat: float, lon: float) -> dict:
    # cache this because points responses don’t change frequently
    return _get_json(f"https://api.weather.gov/points/{lat},{lon}")

# NOTE: keep API route definitions (e.g. @app.get) in a separate module,
# not in the notebook. The notebook should just call points_lookup/_get_json.




<h1>Feature Engineering

Parsing Objects

In [4]:
import numpy as np
import re

def parse_wind_speed(wind_speed: str) -> float:
    """
    Parse the wind speed from the NWS API response.
    Handles formats like "20 mph", "10 to 15 mph", "5-10 mph", etc.
    """
    if not wind_speed:
        return np.nan
    
    # Extract all numeric values using regex (handles "20 mph", "10 to 15", etc.)
    nums = re.findall(r'\d+', str(wind_speed))
    
    if not nums:
        return np.nan
    
    # Convert to integers and return average (handles ranges like "10 to 15")
    nums = [int(n) for n in nums]
    return sum(nums) / len(nums)

# thunderstorm probability
def parse_thunderstorm_probability(probability: str) -> float:
    """
    Parse the thunderstorm probability from the NWS API response.
    """
    if not probability:
        return 0

    keywords = ["thunder", "storm", "t-storm"]
    return int(any(k in str(probability).lower() for k in keywords))

def has_rain(text):
    if not text:
        return 0
    return int("rain" in text.lower())

Categorize Florida's Seasons

In [5]:
def categorize_florida_season(date: Union[pd.Timestamp, str, int]) -> str:
    """
    Categorize a date into Florida's seasonal categories.
    
    Categories:
    - 'dry/mild': November, December, January, February (cooler, drier months)
    - 'spring transition': March, April (transition from dry to wet season)
    - 'wet/thunderstorm': May, June, July (peak wet season with afternoon thunderstorms)
    - 'hurricane peak': August, September, October (peak hurricane season, still wet)
    
    Args:
        date: Can be a pandas Timestamp, datetime string, or month number (1-12)
    
    Returns:
        str: One of 'dry/mild', 'spring transition', 'wet/thunderstorm', 'hurricane peak'
    
    Examples:
        >>> categorize_florida_season(pd.Timestamp('2024-01-15'))
        'dry/mild'
        >>> categorize_florida_season(6)  # June
        'wet/thunderstorm'
    """
    # Handle different input types
    if isinstance(date, (int, float)):
        month = int(date)
    elif isinstance(date, str):
        date = pd.to_datetime(date)
        month = date.month
    elif isinstance(date, pd.Timestamp):
        month = date.month
    else:
        raise ValueError(f"Unsupported date type: {type(date)}")
    
    # Validate month
    if not (1 <= month <= 12):
        raise ValueError(f"Month must be between 1 and 12, got {month}")
    
    # Categorize based on Florida climate patterns
    if month in [11, 12, 1, 2]:  # November, December, January, February
        return 'dry/mild'
    elif month in [3, 4]:  # March, April
        return 'spring transition'
    elif month in [5, 6, 7]:  # May, June, July
        return 'wet/thunderstorm'
    elif month in [8, 9, 10]:  # August, September, October
        return 'hurricane peak'
    else:
        # Should never reach here, but just in case
        raise ValueError(f"Unexpected month value: {month}")

Seasons Features with Forecast

In [6]:
def add_season_features(df: pd.DataFrame, date_column: str = 'startTime') -> pd.DataFrame:
    """
    Add season-related features to a DataFrame with datetime column.
    
    Adds:
    - 'month': Month number (1-12)
    - 'month_sin': Cyclical encoding (sine) for month (12-month cycle)
    - 'month_cos': Cyclical encoding (cosine) for month (12-month cycle)
    - 'florida_season': Categorical season ('dry/mild', 'spring transition', etc.)
    - 'season_numeric': Numeric mapping of season (0-3)
    - 'season_sin': Cyclical encoding (sine) for season (4-season cycle)
    - 'season_cos': Cyclical encoding (cosine) for season (4-season cycle)
    - 'is_dry_season': Binary indicator for dry/mild season
    - 'is_wet_season': Binary indicator for wet/thunderstorm season
    - 'is_hurricane_season': Binary indicator for hurricane peak season
    - 'is_spring_transition': Binary indicator for spring transition
    
    Args:
        df: DataFrame with a datetime column
        date_column: Name of the datetime column (default: 'startTime')
    
    Returns:
        DataFrame with added season features
    """
    df = df.copy()
    
    # Ensure date column is datetime
    if date_column not in df.columns:
        raise ValueError(f"Column '{date_column}' not found in DataFrame")
    
    df[date_column] = pd.to_datetime(df[date_column])
    
    # Extract month
    df['month'] = df[date_column].dt.month
    # Cyclical encoding for months (12-month cycle)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    
    # Categorize season
    df['florida_season'] = df[date_column].apply(categorize_florida_season)
    
    # Map seasons to numeric values for cyclical encoding (4 seasons)
    season_map = {
        'dry/mild': 0,
        'spring transition': 1,
        'wet/thunderstorm': 2,
        'hurricane peak': 3
    }
    df['season_numeric'] = df['florida_season'].map(season_map)
    
    # Cyclical encoding for seasons (4-season cycle)
    df["season_sin"] = np.sin(2 * np.pi * df["season_numeric"] / 4)
    df["season_cos"] = np.cos(2 * np.pi * df["season_numeric"] / 4)
    
    # Create binary indicators for each season
    df['is_dry_season'] = (df['florida_season'] == 'dry/mild').astype(int)
    df['is_spring_transition'] = (df['florida_season'] == 'spring transition').astype(int)
    df['is_wet_season'] = (df['florida_season'] == 'wet/thunderstorm').astype(int)
    df['is_hurricane_season'] = (df['florida_season'] == 'hurricane peak').astype(int)
    
    return df

# convert JSON to pandas DataFrame
def forecast_to_df(json_data: dict) -> pd.DataFrame:
    """
    Convert JSON data to a pandas DataFrame.
    """
    df = pd.DataFrame(json_data)

    # time features
    df['startTime'] = pd.to_datetime(df['startTime'])
    df['hours'] = df['startTime'].dt.hour
    
    # Add season features
    df = add_season_features(df, date_column='startTime')

    # numeric features
    df["wind_mph"] = df["windSpeed"].apply(parse_wind_speed)
    df["precip_prob"] = df["precipChance"].fillna(0) # missing values are 0
    df["humidity"] = df["humidity"].fillna(df["humidity"].mean())

    # Text-derived features
    df["thunderstorm"] = df["shortForecast"].apply(parse_thunderstorm_probability)
    df["rain"] = df["shortForecast"].apply(has_rain)

    # Drop raw text after extraction (optional)
    df = df.drop(columns=["windSpeed", "shortForecast"])

    return df

Weather Indices

In [ ]:
def heat_index(temp_f, humidity):
    # Simple approximation
    return temp_f + 0.1 * humidity
    df['heat_index'] = heat_index(df['tempF'], df['humidity'])

# risk signals
def risk_signals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate risk signals from the DataFrame.
    """
    df['high_rain_risk'] = (df['precip_prob'] > 0.5).astype(int)
    df['high_wind_risk'] = (df['wind_mph'] > 15).astype(int)
    df['heat_wave_risk'] = (df['heat_index'] > 100).astype(int)

    df["hurricane_season"] = df["month"].isin([6,7,8,9,10,11]).astype(int)
    df['precip_risk'] = (df['precip_prob'] * (1 + 0.5 * df['wet_season'])) # higher disruption risk in wet season
    df['seasonal_heat_risk'] = (df['heat_index'] * (1 + 0.3 * df['wet_season'])) # higher humidity amplifies perceived heat, captures nonlinear discomfort
    df['summer_stormisk'] = (df['thunderstorm'] * df['summer']) # higher disruption risk in summer thunderstorms

# training thresholds
def label_weather(row):
    if row["thunderstorm"] == 1:
        return 2  # No-go
    if row["high_rain_risk"] or row["dangerous_heat"]:
        return 1  # Caution
    return 0  # Good

    df['risk_label'] = df.apply(label_weather, axis=1)

# Data Pipeline: From API JSON → Cleaned DataFrame

Let's test the full pipeline: fetch messy JSON from API, then see how it looks after cleaning and feature engineering.

In [ ]:
# Step 1: Fetch data from API (raw JSON - messy!)
import json
import pandas as pd

# Fetch forecast for Miami
city = "Miami"
lat, lon = FL_CITIES[city]

# Get points and hourly forecast
points = points_lookup(lat, lon)
hourly_url = points["properties"]["forecastHourly"]
forecast_data = _get_json(hourly_url)

# Extract the periods (next 24 hours)
raw_periods = forecast_data["properties"]["periods"][:24]

print("=" * 80)
print("RAW JSON DATA (messy, nested structure)")
print("=" * 80)
print(f"\nNumber of periods: {len(raw_periods)}")
print(f"\nFirst period structure (showing keys):")
print(json.dumps(list(raw_periods[0].keys()), indent=2))
print(f"\nSample of first period (pretty-printed):")
print(json.dumps(raw_periods[0], indent=2)[:1000] + "...")

RAW JSON DATA (messy, nested structure)

Number of periods: 24

First period structure (showing keys):
[
  "number",
  "name",
  "startTime",
  "endTime",
  "isDaytime",
  "temperature",
  "temperatureUnit",
  "temperatureTrend",
  "probabilityOfPrecipitation",
  "dewpoint",
  "relativeHumidity",
  "windSpeed",
  "windDirection",
  "icon",
  "shortForecast",
  "detailedForecast"
]

Sample of first period (pretty-printed):
{
  "number": 1,
  "name": "",
  "startTime": "2026-01-27T16:00:00-05:00",
  "endTime": "2026-01-27T17:00:00-05:00",
  "isDaytime": true,
  "temperature": 67,
  "temperatureUnit": "F",
  "temperatureTrend": null,
  "probabilityOfPrecipitation": {
    "unitCode": "wmoUnit:percent",
    "value": 0
  },
  "dewpoint": {
    "unitCode": "wmoUnit:degC",
    "value": 11.11111111111111
  },
  "relativeHumidity": {
    "unitCode": "wmoUnit:percent",
    "value": 59
  },
  "windSpeed": "15 mph",
  "windDirection": "N",
  "icon": "https://api.weather.gov/icons/land/day/few?size=

In [9]:
# Step 2: Convert to simplified JSON format (what your API returns)
simplified_data = {
    "city": city,
    "lat": lat,
    "lon": lon,
    "hours": [{
        "startTime": p["startTime"],
        "tempF": p["temperature"],
        "windSpeed": p["windSpeed"],
        "windDirection": p.get("windDirection"),
        "shortForecast": p.get("shortForecast"),
        "precipChance": (p.get("probabilityOfPrecipitation") or {}).get("value"),
        "humidity": (p.get("relativeHumidity") or {}).get("value"),
    } for p in raw_periods]
}

print("=" * 80)
print("SIMPLIFIED JSON (after cleaning - what your API endpoint returns)")
print("=" * 80)
print(f"\nStructure: {list(simplified_data.keys())}")
print(f"\nFirst hour's data:")
print(json.dumps(simplified_data["hours"][0], indent=2))

SIMPLIFIED JSON (after cleaning - what your API endpoint returns)

Structure: ['city', 'lat', 'lon', 'hours']

First hour's data:
{
  "startTime": "2026-01-27T16:00:00-05:00",
  "tempF": 67,
  "windSpeed": "15 mph",
  "windDirection": "N",
  "shortForecast": "Sunny",
  "precipChance": 0,
  "humidity": 59
}


In [10]:
# Step 3: Convert to DataFrame and apply feature engineering
# This is what your API users would do with the JSON response

df_raw = pd.DataFrame(simplified_data["hours"])
print("=" * 80)
print("INITIAL DATAFRAME (after converting JSON to pandas)")
print("=" * 80)
print(f"\nShape: {df_raw.shape} (rows × columns)")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nData types:")
print(df_raw.dtypes)
print(f"\nFirst few rows (before feature engineering):")
display(df_raw.head())

INITIAL DATAFRAME (after converting JSON to pandas)

Shape: (24, 7) (rows × columns)

Columns: ['startTime', 'tempF', 'windSpeed', 'windDirection', 'shortForecast', 'precipChance', 'humidity']

Data types:
startTime        object
tempF             int64
windSpeed        object
windDirection    object
shortForecast    object
precipChance      int64
humidity          int64
dtype: object

First few rows (before feature engineering):


,startTime,tempF,windSpeed,windDirection,shortForecast,precipChance,humidity
0,2026-01-27T16:00:00-05:00,67,15 mph,N,Sunny,0,59
1,2026-01-27T17:00:00-05:00,66,15 mph,N,Sunny,0,61
2,2026-01-27T18:00:00-05:00,65,14 mph,N,Mostly Clear,0,60
3,2026-01-27T19:00:00-05:00,64,14 mph,N,Mostly Clear,0,60
4,2026-01-27T20:00:00-05:00,62,14 mph,N,Mostly Clear,0,65


In [11]:
# Step 4: Apply feature engineering pipeline
df_cleaned = forecast_to_df(simplified_data["hours"])

print("=" * 80)
print("FINAL CLEANED DATAFRAME (after feature engineering)")
print("=" * 80)
print(f"\nShape: {df_cleaned.shape} (rows × columns)")
print(f"\nAll columns ({len(df_cleaned.columns)} total):")
for i, col in enumerate(df_cleaned.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n{'='*80}")
print("FULL DATAFRAME VIEW (like CSV/Excel - tabular format)")
print("=" * 80)
display(df_cleaned)

FINAL CLEANED DATAFRAME (after feature engineering)

Shape: (24, 21) (rows × columns)

All columns (21 total):
   1. startTime
   2. tempF
   3. windDirection
   4. precipChance
   5. humidity
   6. hours
   7. month
   8. month_sin
   9. month_cos
  10. florida_season
  11. season_numeric
  12. season_sin
  13. season_cos
  14. is_dry_season
  15. is_spring_transition
  16. is_wet_season
  17. is_hurricane_season
  18. wind_mph
  19. precip_prob
  20. thunderstorm
  21. rain

FULL DATAFRAME VIEW (like CSV/Excel - tabular format)


,startTime,tempF,windDirection,precipChance,humidity,hours,month,month_sin,month_cos,florida_season,...,season_sin,season_cos,is_dry_season,is_spring_transition,is_wet_season,is_hurricane_season,wind_mph,precip_prob,thunderstorm,rain
0,2026-01-27 16:00:00-05:00,67,N,0,59,16,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,15.0,0,0,0
1,2026-01-27 17:00:00-05:00,66,N,0,61,17,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,15.0,0,0,0
2,2026-01-27 18:00:00-05:00,65,N,0,60,18,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,0,0,0
3,2026-01-27 19:00:00-05:00,64,N,0,60,19,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,0,0,0
4,2026-01-27 20:00:00-05:00,62,N,0,65,20,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,0,0,0
5,2026-01-27 21:00:00-05:00,61,N,0,65,21,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,0,0,0
6,2026-01-27 22:00:00-05:00,60,N,1,67,22,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,1,0,0
7,2026-01-27 23:00:00-05:00,59,N,1,69,23,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,1,0,0
8,2026-01-28 00:00:00-05:00,59,N,1,72,0,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,1,0,0
9,2026-01-28 01:00:00-05:00,58,N,2,75,1,1,0.5,0.866025,dry/mild,...,0.0,1.0,1,0,0,0,14.0,2,0,0


In [12]:
# Step 5: Show summary statistics (like Excel would show)
print("=" * 80)
print("SUMMARY STATISTICS (like CSV/Excel summary)")
print("=" * 80)
print("\nNumeric columns summary:")
display(df_cleaned.describe())

print("\n" + "="*80)
print("Categorical columns:")
print("=" * 80)
if 'florida_season' in df_cleaned.columns:
    print("\nSeason distribution:")
    print(df_cleaned['florida_season'].value_counts())

print("\n" + "="*80)
print("Sample of key engineered features:")
print("=" * 80)
feature_cols = ['startTime', 'tempF', 'wind_mph', 'precip_prob', 'humidity', 
                'florida_season', 'month_sin', 'month_cos', 
                'season_sin', 'season_cos', 'thunderstorm', 'rain']
available_features = [col for col in feature_cols if col in df_cleaned.columns]
display(df_cleaned[available_features].head(10))

SUMMARY STATISTICS (like CSV/Excel summary)

Numeric columns summary:


,tempF,precipChance,humidity,hours,month,month_sin,month_cos,season_numeric,season_sin,season_cos,is_dry_season,is_spring_transition,is_wet_season,is_hurricane_season,wind_mph,precip_prob,thunderstorm,rain
count,24.000000,24.000000,24.000000,24.000000,24.0,24.0,24.000000,24.0,24.0,24.0,24.0,24.0,24.0,24.0,24.000000,24.000000,24.0,24.0
mean,60.875000,3.875000,71.083333,11.500000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,13.500000,3.875000,0.0,0.0
std,3.848348,4.099973,6.665398,7.071068,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.978019,4.099973,0.0,0.0
min,56.000000,0.000000,59.000000,0.000000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,12.000000,0.000000,0.0,0.0
25%,57.000000,0.750000,66.500000,5.750000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,12.750000,0.750000,0.0,0.0
50%,60.500000,2.500000,72.000000,11.500000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,14.000000,2.500000,0.0,0.0
75%,64.250000,6.500000,77.000000,17.250000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,14.000000,6.500000,0.0,0.0
max,67.000000,12.000000,80.000000,23.000000,1.0,0.5,0.866025,0.0,0.0,1.0,1.0,0.0,0.0,0.0,15.000000,12.000000,0.0,0.0



Categorical columns:

Season distribution:
florida_season
dry/mild    24
Name: count, dtype: int64

Sample of key engineered features:


,startTime,tempF,wind_mph,precip_prob,humidity,florida_season,month_sin,month_cos,season_sin,season_cos,thunderstorm,rain
0,2026-01-27 16:00:00-05:00,67,15.0,0,59,dry/mild,0.5,0.866025,0.0,1.0,0,0
1,2026-01-27 17:00:00-05:00,66,15.0,0,61,dry/mild,0.5,0.866025,0.0,1.0,0,0
2,2026-01-27 18:00:00-05:00,65,14.0,0,60,dry/mild,0.5,0.866025,0.0,1.0,0,0
3,2026-01-27 19:00:00-05:00,64,14.0,0,60,dry/mild,0.5,0.866025,0.0,1.0,0,0
4,2026-01-27 20:00:00-05:00,62,14.0,0,65,dry/mild,0.5,0.866025,0.0,1.0,0,0
5,2026-01-27 21:00:00-05:00,61,14.0,0,65,dry/mild,0.5,0.866025,0.0,1.0,0,0
6,2026-01-27 22:00:00-05:00,60,14.0,1,67,dry/mild,0.5,0.866025,0.0,1.0,0,0
7,2026-01-27 23:00:00-05:00,59,14.0,1,69,dry/mild,0.5,0.866025,0.0,1.0,0,0
8,2026-01-28 00:00:00-05:00,59,14.0,1,72,dry/mild,0.5,0.866025,0.0,1.0,0,0
9,2026-01-28 01:00:00-05:00,58,14.0,2,75,dry/mild,0.5,0.866025,0.0,1.0,0,0


<h1> Train/Test Split

In [13]:
# Import risk scoring functions from feature_utils
# Force reload to get latest version (important after code changes!)
import importlib
import src.features.feature_utils
importlib.reload(src.features.feature_utils)
from src.features.feature_utils import risk_signals, build_cost_based_risk_score, score_to_label, heat_index

# Start with cleaned dataframe (assuming df_cleaned exists from previous cells)
df = df_cleaned.sort_values("startTime").reset_index(drop=True)

# Step 1: Ensure heat_index exists (required for risk scoring)
if 'heat_index' not in df.columns and 'tempF' in df.columns and 'humidity' in df.columns:
    df['heat_index'] = heat_index(df['tempF'], df['humidity'])

# Step 2: Build risk signals (creates high_rain_risk, high_wind_risk, heat_wave_risk, etc.)
df = risk_signals(df)

# Step 3: Build cost-based risk score (0-100)
df = build_cost_based_risk_score(df)

# Step 4: Convert risk score to labels (0=Good, 1=Caution, 2=No-go)
df['risk_label'] = df['risk_score_0_100'].apply(score_to_label)

print("=" * 80)
print("RISK SCORING COMPLETE")
print("=" * 80)
print(f"\nRisk Score Statistics:")
print(df[['risk_score_0_100', 'risk_label']].describe())
print(f"\nLabel Distribution:")
print(df['risk_label'].value_counts().sort_index())
print(f"\nLabel meanings: 0=Good, 1=Caution, 2=No-go")
print(f"\nSample of risk scores and labels:")
display(df[['startTime', 'risk_score_0_100', 'risk_label', 'florida_season']].head(10))

RISK SCORING COMPLETE

Risk Score Statistics:
       risk_score_0_100  risk_label
count         24.000000        24.0
mean           0.700000         0.0
std            0.195604         0.0
min            0.400000         0.0
25%            0.550000         0.0
50%            0.800000         0.0
75%            0.800000         0.0
max            1.000000         0.0

Label Distribution:
risk_label
0    24
Name: count, dtype: int64

Label meanings: 0=Good, 1=Caution, 2=No-go

Sample of risk scores and labels:


,startTime,risk_score_0_100,risk_label,florida_season
0,2026-01-27 16:00:00-05:00,1.0,0,dry/mild
1,2026-01-27 17:00:00-05:00,1.0,0,dry/mild
2,2026-01-27 18:00:00-05:00,0.8,0,dry/mild
3,2026-01-27 19:00:00-05:00,0.8,0,dry/mild
4,2026-01-27 20:00:00-05:00,0.8,0,dry/mild
5,2026-01-27 21:00:00-05:00,0.8,0,dry/mild
6,2026-01-27 22:00:00-05:00,0.8,0,dry/mild
7,2026-01-27 23:00:00-05:00,0.8,0,dry/mild
8,2026-01-28 00:00:00-05:00,0.8,0,dry/mild
9,2026-01-28 01:00:00-05:00,0.8,0,dry/mild


In [14]:
# Train/Test Split (temporal split)
# Note: df should already have risk_label from previous cell
train_df = df[df["startTime"] < "2025-01-01"].copy()
test_df  = df[df["startTime"] >= "2025-01-01"].copy()

print(f"{'='*80}")
print(f"TRAIN/TEST SPLIT")
print(f"{'='*80}")
print(f"Train set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test set: {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"\nTrain label distribution:")
print(train_df['risk_label'].value_counts().sort_index())
print(f"\nTest label distribution:")
print(test_df['risk_label'].value_counts().sort_index())

# Prepare features and labels for ML
# Select feature columns (exclude target and metadata)
feature_cols = [col for col in df.columns if col not in [
    'risk_label', 'risk_score_0_100', 'risk_raw', 'risk_raw_adj',
    'precip_severity_0_1', 'heat_severity_0_1', 'wind_severity_0_1',
    'startTime', 'florida_season'  # exclude datetime and categorical
]]

X_train = train_df[feature_cols]
y_train = train_df["risk_label"]

X_test = test_df[feature_cols]
y_test = test_df["risk_label"]

print(f"\n{'='*80}")
print(f"FEATURE PREPARATION")
print(f"{'='*80}")
print(f"Number of features: {len(feature_cols)}")
if len(feature_cols) > 10:
    print(f"First 10 features: {feature_cols[:10]}")
    print(f"... and {len(feature_cols) - 10} more")
else:
    print(f"All features: {feature_cols}")


TRAIN/TEST SPLIT
Train set: 0 samples (0.0%)
Test set: 24 samples (100.0%)

Train label distribution:
Series([], Name: count, dtype: int64)

Test label distribution:
risk_label
0    24
Name: count, dtype: int64

FEATURE PREPARATION
Number of features: 28
First 10 features: ['tempF', 'windDirection', 'precipChance', 'humidity', 'hours', 'month', 'month_sin', 'month_cos', 'season_numeric', 'season_sin']
... and 18 more


<h1>Percentage-Based Split

In [15]:
df = df.sort_values("startTime").reset_index(drop=True)

split_frac = 0.7  # 70% train, 30% test
split_idx = int(len(df) * split_frac)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

print(f"Train rows: {len(train_df)} ({len(train_df)/len(df):.1%})")
print(f"Test rows:  {len(test_df)} ({len(test_df)/len(df):.1%})")

print("\nTrain label distribution:")
print(train_df["risk_label"].value_counts(dropna=False))

print("\nTest label distribution:")
print(test_df["risk_label"].value_counts(dropna=False))

Train rows: 16 (66.7%)
Test rows:  8 (33.3%)

Train label distribution:
risk_label
0    16
Name: count, dtype: int64

Test label distribution:
risk_label
0    8
Name: count, dtype: int64


In [16]:
# Keep only available columns (prevents KeyErrors)
available_features = [c for c in feature_cols if c in df.columns]

# Drop any non-numeric columns just in case
X_train = train_df[available_features].select_dtypes(include="number")
X_test  = test_df[available_features].select_dtypes(include="number")

y_train = train_df["risk_label"]
y_test  = test_df["risk_label"]

print("Num features used:", X_train.shape[1])
print("Dropped non-numeric:", set(available_features) - set(X_train.columns))

Num features used: 27
Dropped non-numeric: {'windDirection'}


In [21]:
def make_dev_labels_ranked(df, score_col="risk_score_0_100"):
    df = df.copy()
    df["_rank"] = df[score_col].rank(method="first")

    n = len(df)
    # bottom 50% → Good
    # next 30% → Caution
    # top 20% → No-go
    df["risk_label"] = np.select(
        [
            df["_rank"] > 0.8 * n,
            df["_rank"] > 0.5 * n
        ],
        [2, 1],
        default=0
    )

    df.drop(columns="_rank", inplace=True)
    return df

df = make_dev_labels_ranked(df)

print(df["risk_label"].value_counts().sort_index())


risk_label
0    12
1     7
2     5
Name: count, dtype: int64


In [22]:
df = df.sort_values("startTime").reset_index(drop=True)

split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

print("Train label distribution:\n", train_df["risk_label"].value_counts().sort_index())
print("Test label distribution:\n", test_df["risk_label"].value_counts().sort_index())

Train label distribution:
 risk_label
0    4
1    7
2    5
Name: count, dtype: int64
Test label distribution:
 risk_label
0    8
Name: count, dtype: int64


<h1> Train Gradient Boosting

In [23]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train = train_df[available_features].select_dtypes(include="number")
y_train = train_df["risk_label"]

X_test = test_df[available_features].select_dtypes(include="number")
y_test = test_df["risk_label"]

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       8.0
           2       0.00      0.00      0.00       0.0

    accuracy                           0.00       8.0
   macro avg       0.00      0.00      0.00       8.0
weighted avg       0.00      0.00      0.00       8.0

[[0 8]
 [0 0]]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

<h1> Force Split into 2 Classes

In [25]:
from sklearn.model_selection import train_test_split

X = df[available_features].select_dtypes(include="number")
y = df["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.33,
    random_state=42,
    stratify=y   # ensures both classes appear in train/test
)

In [26]:
# label distribution

print("Train:", y_train.value_counts().sort_index())
print("Test:", y_test.value_counts().sort_index())

Train: risk_label
0    8
1    5
2    3
Name: count, dtype: int64
Test: risk_label
0    4
1    2
2    2
Name: count, dtype: int64


<h1> Confusion Matrix

In [27]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

[[0 0 4]
 [0 0 2]
 [0 0 2]]


## Summary: Data Flow Visualization

**Raw JSON** (messy, nested) → **Simplified JSON** (flat structure) → **DataFrame** → **Feature Engineering** → **Final DataFrame**

The final output looks **exactly like CSV/Excel** - it's a tabular DataFrame with rows (observations) and columns (features). You can:
- Save it to CSV: `df_cleaned.to_csv('forecast_features.csv', index=False)`
- Save to Excel: `df_cleaned.to_excel('forecast_features.xlsx', index=False)`
- Use directly in ML models (sklearn, etc.)